In [ ]:
# class ReinforcementLearningDemo:
#     def __init__(self):
#         self.score = 0
#         self.game_over = False

#     def cartpole_example(self):
#         print("목표 : 막대 쓰러뜨리지 않기")

#         while not self.game_over:
#             state = {
#                 'pole_angle': 0.1,      
#                 'cart_position': 0.0,   
#                 'pole_velocity': 0.02,  
#                 'cart_velocity': 0.1    
#             }
#             print(state)

#             action = self.choose_action(state)  
#             print(f"선택한 행동: {'왼쪽' if action == -1 else '오른쪽'}")

#             reward = self.calculate_reward(state, action)
#             print(f"받은 보상: {reward}")

#             self.update_policy(state, action, reward)
#             print("전략 업데이트")

#             self.score += reward
#             if self.score < -100:  
#                 self.game_over = True

#         print(f"최종 스코어: {self.score}")

#     def choose_action(self, state):
#         import random

#         if random.random() < 0.9:
            
#             return 1 if state['pole_angle'] > 0 else -1
#         else:
#             return random.choice([-1, 1])  

#     def calculate_reward(self, state, action):
#         if abs(state['pole_angle']) > 0.5:  
#             return -100  
#         else:
#             return +1    

#     def update_policy(self, state, action, reward):
#         learning_rate = 0.1
#         discount_factor = 0.95

#         print("   더 나은 전략으로 업데이트!")

# demo = ReinforcementLearningDemo()
# demo.cartpole_example()

In [ ]:
import gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

class QNetwork(nn.Module):
    def __init__(self, state_size, action_size, seed, fc1_units=64, fc2_units=64):
        super(QNetwork, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, fc2_units)
        self.fc3 = nn.Linear(fc2_units, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
seed = 1234
qnetwork_local = QNetwork(state_size, action_size, seed)
qnetwork_target = QNetwork(state_size, action_size, seed)
optimizer = optim.Adam(qnetwork_local.parameters(), lr=5e-4)

buffer_size = int(1e5)
batch_size = 64
memory = deque(maxlen= buffer_size)

def step(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def sample():
    experiences = random.sample(memory, k=batch_size)
    states = torch.from_numpy(np.vstack([e[0] for e in experiences])).float()
    actions = torch.from_numpy(np.vstack([e[1] for e in experiences])).long()
    rewards = torch.from_numpy(np.vstack([e[2] for e in experiences])).float()
    next_states = torch.from_numpy(np.vstack([e[3] for e in experiences])).float()
    dones = torch.from_numpy(np.vstack([e[4] for e in experiences]).astype(np.uint8)).float()
    return (states, actions, rewards, next_states, dones)

def learn(experiences, gamma):
    states, actions, rewards, next_states, dones = experiences
    Q_targets_next = qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
    Q_targets = rewards + (gamma * Q_targets_next * (1 - dones))
    Q_expected = qnetwork_local(states).gather(1, actions)
    loss = nn.MSELoss()(Q_expected, Q_targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

gamma = 0.99
tau = 1e-3
n_episodes = 200
max_t = 1000

for i_episode in range(1, n_episodes+1):
    state = env.reset()
    total_reward = 0
    for t in range(max_t):
        state_tensor = torch.from_numpy(state).float().unsqueeze(0)
        with torch.no_grad():
            action_values = qnetwork_local(state_tensor)
        action = np.argmax(action_values.cpu().data.numpy())
        next_state, reward, done, _ = env.step(action)
        step(state, action, reward, next_state, done)
        total_reward += reward
        if len(memory) > batch_size:
            experiences = sample()
            loss = learn(experiences, gamma)
        state = next_state
        if done:
            break
    
    for target_param, local_param in zip(qnetwork_target.parameters(), qnetwork_local.parameters()):
        target_param.data.copy_(tau*local_param.data + (1-tau)*target_param.data)
    print(i_episode, total_reward)

env.close()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = self.fc2(x)
        return torch.softmax(x, dim=-1)

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

policy_net = PolicyNetwork(state_size, action_size)
optimizer = optim.Adam(policy_net.parameters(), lr=0.01)

def select_action(state):
    state = torch.from_numpy(state).float().unsqueeze(0)
    probs = policy_net(state)
    action = torch.multinomial(probs, num_samples=1)
    return action.item(), torch.log(probs[0, action.item()])

def reinforce_update(episode_rewards, episode_log_probs, gamma=0.99):
    R = 0
    returns = []

    for r in episode_rewards[::-1]:
        R = r + gamma * R
        returns.insert(0, R)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    loss = 0
    for log_prob, R in zip(episode_log_probs, returns):
        loss -= log_prob * R
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

num_episodes = 1000
for episode in range(num_episodes):
    state = env.reset()
    episode_rewards = []
    episode_log_probs = []
    done = False
    while not done:
        action, log_prob = select_action(state)
        next_state, reward, done, _ = env.step(action)
        episode_rewards.append(reward)
        episode_log_probs.append(log_prob)
        state = next_state
    loss = reinforce_update(episode_rewards, episode_log_probs)
    if episode % 50 == 0:
        total_reward = sum(episode_rewards)
        print(f"Episode {episode}, Total Reward: {total_reward}, Loss: {loss:.3f}")

env.close()